# 05 - The Five Agent Tools

**Learning objective:** Understand and demo each of the five tools the ReAct agent will use.

Tools:
1. `search_drug_kb` - Embedding search across all chunks
2. `lookup_interactions` - Targeted drug interaction retrieval
3. `lookup_population_warnings` - Population-specific warnings
4. `flag_severity` - Classify findings by severity
5. `summarize_evidence` - Produce the final cited report

In [ ]:
import os
import sys
import subprocess
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

# Connect to database
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)

# Embeddings client
mi_dir = "../iac_snippets/managed_inference"
result = subprocess.run(["tofu", "output", "-raw", "endpoint_url"], cwd=mi_dir, capture_output=True, text=True)
endpoint_url = result.stdout.strip()

from src.embeddings import EmbeddingsClient

embedding_client = EmbeddingsClient(
    client=OpenAI(base_url=endpoint_url, api_key=os.environ["SCW_SECRET_KEY"]),
    model="baai/bge-multilingual-gemma2:fp16",
)

# LLM client
llm_client = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

# Create toolkit
from src.tools import ToolKit

toolkit = ToolKit(conn=conn, embeddings_client=embedding_client, llm_client=llm_client)

print("Toolkit ready.")

In [ ]:
# Tool 1: search_drug_kb
results = toolkit.search_drug_kb("bleeding risk anticoagulant")

print("search_drug_kb('bleeding risk anticoagulant')")
print("---")
for r in results:
    print(f"  {r['drug_name']} | {r['section_type']} | {r.get('distance', 'N/A')}")
    print(f"  {r['text'][:120]}...")
    print()

In [ ]:
# Tool 2: lookup_interactions - warfarin
results = toolkit.lookup_interactions("warfarin")

print("lookup_interactions('warfarin')")
print("---")
for r in results:
    print(f"  Section: {r['section_type']} | Set ID: {r.get('set_id', 'N/A')}")
    print(f"  {r['text'][:200]}...")
    print()

In [ ]:
# Tool 2: lookup_interactions - clarithromycin
results = toolkit.lookup_interactions("clarithromycin")

print("lookup_interactions('clarithromycin')")
print("---")
for r in results:
    print(f"  Section: {r['section_type']} | Set ID: {r.get('set_id', 'N/A')}")
    print(f"  {r['text'][:200]}...")
    print()

In [ ]:
# Tool 3: lookup_population_warnings - ibuprofen pregnancy
results = toolkit.lookup_population_warnings("ibuprofen", "pregnancy")

print("lookup_population_warnings('ibuprofen', 'pregnancy')")
print("---")
for r in results:
    print(f"  Section: {r['section_type']}")
    print(f"  {r['text'][:200]}...")
    print()

In [ ]:
# Tool 3: lookup_population_warnings - metformin renal
results = toolkit.lookup_population_warnings("metformin", "renal")

print("lookup_population_warnings('metformin', 'renal')")
print("---")
for r in results:
    print(f"  Section: {r['section_type']}")
    print(f"  {r['text'][:200]}...")
    print()

In [ ]:
# Tools 4 & 5: flag_severity and summarize_evidence
sample_findings = [
    {
        "claim": "Warfarin + aspirin increases bleeding risk",
        "source_section_type": "drug_interactions",
        "source_id": "WARFARIN SODIUM :: drug_interactions :: e98a2d84-c192-4d2a-b202-61a81b0c7dda",
        "evidence_snippet": "Drugs that Increase Bleeding Risk: aspirin, ibuprofen, NSAIDs",
    },
    {
        "claim": "Warfarin can cause major or fatal bleeding",
        "source_section_type": "boxed_warning",
        "source_id": "WARFARIN SODIUM :: boxed_warning :: e98a2d84-c192-4d2a-b202-61a81b0c7dda",
        "evidence_snippet": "WARNING: BLEEDING RISK - Warfarin sodium can cause major or fatal bleeding.",
    },
]

flagged = toolkit.flag_severity(sample_findings)
print("flag_severity results (should be severity-ordered):")
for f in flagged:
    print(f"  [{f['severity']}] {f['claim']}")

print()

summary = toolkit.summarize_evidence(flagged)
print("summarize_evidence results:")
print(json.dumps(summary, indent=2))

## Side-by-side: Raw Mistral vs Tool-Grounded

| Aspect | Raw Mistral (notebook 01) | Tool-Grounded (this notebook) |
|--------|--------------------------|-------------------------------|
| Source | Model's training data | Specific FDA label sections |
| Citations | None | `<drug> :: <section> :: <set_id>` |
| Verifiable | No | Yes (via openFDA/DailyMed) |
| Severity | Implicit | Explicit (CRITICAL/MAJOR/MODERATE/MINOR) |

## You should now have

- [x] Tested all five tools individually
- [x] Seen grounded citations with set_id references
- [x] Understood severity classification (boxed_warning = CRITICAL)

**Next:** `06_react_agent.ipynb`